# Datamine GAUSANAM Process Example

This notebook demonstrates how to configure and run the **`gausanam`** process wrapper in `dmstudio`.

## Process Description

## GAUSANAM

For mosaic-style modelling, all variograms are directly proportional, so indicator kriging will provide a correct solution. Diffusion-style models are a bigger challenge in that [co-kriging](<../STUDIO_RM/Grade%20Estimate%20Overview.md#cokriging>) is required as the model is discretely gaussian in nature. The important point is that the gaussian model will retain a gaussian distribution when the [support change](<../Uniform_Conditioning/About_Change_of_Support.md>) occurs (from points to blocks) as a result of Gaussian Anamorphosis.

The gaussian anamorphosis is a mathematical function which transforms a variable Y with a gaussian distribution in a new variable Z with any distribution. From a pragmatic viewpoint, this means transforming a non-Gaussian distribution into a Gaussian distribution (anamorphosis means transformation).

![](../Images/unicon8_461x229.gif)

The Gaussian Anamorphosis Modelling functionality is designed to:

* model the histogram of your raw dataset (Anamorphosis function),
* transform a Raw Variable into a Gaussian Variable (normal score transformation) that will be used in the simulation processes, and;
* calculate grade-tonnage curves

### Input Files:

* **samples** (Undefined):
  A Datamine file that contains sample positional information and supporting attributes.
  Required=Yes

### Output Files:

* **graph** (Undefined):
  A file containing the data required to construct scatter plot and histogram graphs
  relating to a locally-conditioned SMU model.
  Required=No

* **stats** (Undefined):
  A file containing summary statistical data (in Datamine binary format) relating to a
  locally-conditioned SMU model.
  Required=No

### Fields:

* **grade** (Alphanumeric : IN):
  The grade field (present in the samples file) that will be considered during
  anamorphosis.
  Default=Undefined
  Required=Yes

* **weight** (Alphanumeric : IN):
  An optional weighting field.
  Default=Undefined
  Required=No

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('gausanam')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute gausanam
print("Running gausanam...")
dm_cmd.gausanam(
    samples_i=['t_assays'],  # required
    stats_o=['t_gausanam_out'],  # required
    grade_f='optional',  # required
    # graph_o='t_gausanam_out',  # optional
    # weight_f='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("gausanam execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_gausanam_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")